In [29]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("LLM_API_KEY")
print(api_key[:10])

sk-or-v1-d


In [30]:
import requests

url = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    payload = {
        "model": "poolside/laguna-xs-2.1:free",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Status Code:", response.status_code)
        print(response.text)
        return None

    data = response.json()

    # Print the full response for debugging (optional)
    # print(data)

    try:
        content = data["choices"][0]["message"]["content"]

        if content is None:
            print("Model returned no content.")
            return None

        return content.strip()

    except Exception as e:
        print("Error parsing response:", e)
        print(data)
        return None

In [31]:
system_prompt = "You are a helpful assistant"

user_prompt = "Reply with only the word: hello"

result = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(result)

hello


In [32]:
import joblib
best_model = joblib.load("../part3/best_model.pkl")
print(best_model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(n_estimators=200, random_state=42))])


In [33]:
sample1 = {
    "age":25,
    "bmi":22.5,
    "children":0,
    "blood_pressure":120,
    "exercise_frequency":3,
    "pre_existing_condition":False,
    "occupation_risk":0,
    "annual_income":45000,
    "sex_male":1,
    "smoker_yes":0,
    "region_northwest":0,
    "region_southeast":0,
    "region_southwest":1
}

sample2 = {
    "age":55,
    "bmi":34.2,
    "children":2,
    "blood_pressure":145,
    "exercise_frequency":1,
    "pre_existing_condition":True,
    "occupation_risk":2,
    "annual_income":85000,
    "sex_male":0,
    "smoker_yes":1,
    "region_northwest":1,
    "region_southeast":0,
    "region_southwest":0
}

sample3 = {
    "age":40,
    "bmi":29.4,
    "children":1,
    "blood_pressure":130,
    "exercise_frequency":2,
    "pre_existing_condition":False,
    "occupation_risk":1,
    "annual_income":65000,
    "sex_male":1,
    "smoker_yes":0,
    "region_northwest":0,
    "region_southeast":1,
    "region_southwest":0
}

In [34]:
import pandas as pd
samples = [sample1, sample2, sample3]
for sample in samples:

    df = pd.DataFrame([sample])
    pred = best_model.predict(df)[0]
    prob = best_model.predict_proba(df)[0][pred]
    print(pred, prob)

0 0.88
1 0.955
0 0.905


In [35]:
from jsonschema import validate, ValidationError

explanation_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {
            "type": "string"
        },
        "confidence_level": {
            "type": "string"
        },
        "top_reason": {
            "type": "string"
        },
        "second_reason": {
            "type": "string"
        },
        "next_step": {
            "type": "string"
        }
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

In [36]:
system_prompt = """
You are an AI assistant that explains machine learning predictions.

Return ONLY valid JSON.

Do not include markdown.
Do not include code blocks.
Do not include explanations.
Do not include any text before or after the JSON.

The JSON must contain exactly these fields:

{
  "prediction_label": "Low Risk or High Risk",
  "confidence_level": "Low, Medium, or High",
  "top_reason": "Main reason for the prediction",
  "second_reason": "Second most important reason",
  "next_step": "Recommended action"
}
"""

In [37]:
def create_prompt(features, pred, prob):

    return f"""
Feature values:
{features}

Predicted class: {pred}

Prediction probability: {prob:.3f}

Explain why the model made this prediction.

Return ONLY the JSON object.
"""

In [38]:
import json
from jsonschema import validate, ValidationError

response = call_llm(
    system_prompt,
    create_prompt(sample1, 0, 0.88),
    temperature=0
)

print("Raw Response:")
print(response)

if response is not None:

    try:

        parsed = json.loads(response)

        validate(
            instance=parsed,
            schema=explanation_schema
        )

        print("Validation Passed")
        print(parsed)

    except json.JSONDecodeError as e:

        print("JSON Decode Error:")
        print(e)

    except ValidationError as e:

        print("Schema Validation Error:")
        print(e)

else:

    print("LLM returned no response.")

Raw Response:
{
  "prediction_label": "Low Risk",
  "confidence_level": "High",
  "top_reason": "Absence of pre-existing conditions and non-smoking status significantly reduce health risk factors",
  "second_reason": "Healthy BMI (22.5), normal blood pressure (120), and regular exercise frequency (3 times weekly) indicate good physical health",
  "next_step": "Continue maintaining current healthy lifestyle habits and regular health monitoring"
}
Validation Passed
{'prediction_label': 'Low Risk', 'confidence_level': 'High', 'top_reason': 'Absence of pre-existing conditions and non-smoking status significantly reduce health risk factors', 'second_reason': 'Healthy BMI (22.5), normal blood pressure (120), and regular exercise frequency (3 times weekly) indicate good physical health', 'next_step': 'Continue maintaining current healthy lifestyle habits and regular health monitoring'}


In [39]:
import json
from jsonschema import validate, ValidationError

samples = [sample1, sample2, sample3]

for i, sample in enumerate(samples, start=1):

    print(f"\nSample {i}")

    # Convert sample to DataFrame
    sample_df = pd.DataFrame([sample])

    # Model prediction
    predicted_class = best_model.predict(sample_df)[0]
    predicted_probability = best_model.predict_proba(sample_df)[0][predicted_class]

    print("Input Features:")
    print(sample)

    print("\nPredicted Class:", predicted_class)
    print(f"Predicted Probability: {predicted_probability:.3f}")

    # Build prompt
    user_prompt = create_prompt(
        sample,
        predicted_class,
        predicted_probability
    )

    # Call LLM
    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print("\nRaw LLM Response:")
    print(response)

    # Validate JSON
    if response is not None:

        try:

            parsed = json.loads(response.strip())

            validate(
                instance=parsed,
                schema=explanation_schema
            )

            print("\nValidation Status: PASS")

        except json.JSONDecodeError as e:

            print("\nValidation Status: FAIL")
            print("JSON Decode Error:", e)

        except ValidationError as e:

            print("\nValidation Status: FAIL")
            print("Schema Validation Error:", e.message)

    else:

        print("\nValidation Status: FAIL (No response)")


Sample 1
Input Features:
{'age': 25, 'bmi': 22.5, 'children': 0, 'blood_pressure': 120, 'exercise_frequency': 3, 'pre_existing_condition': False, 'occupation_risk': 0, 'annual_income': 45000, 'sex_male': 1, 'smoker_yes': 0, 'region_northwest': 0, 'region_southeast': 0, 'region_southwest': 1}

Predicted Class: 0
Predicted Probability: 0.880

Raw LLM Response:
{
  "prediction_label": "Low Risk",
  "confidence_level": "High",
  "top_reason": "Absence of pre-existing conditions and non-smoking status significantly reduce health risk factors",
  "second_reason": "Normal blood pressure (120) and healthy BMI (22.5) indicate optimal cardiovascular and metabolic health",
  "next_step": "Continue maintaining healthy lifestyle habits and schedule regular preventive health screenings"
}

Validation Status: PASS

Sample 2
Input Features:
{'age': 55, 'bmi': 34.2, 'children': 2, 'blood_pressure': 145, 'exercise_frequency': 1, 'pre_existing_condition': True, 'occupation_risk': 2, 'annual_income': 850

In [40]:
samples = [sample1, sample2, sample3]

for i, sample in enumerate(samples, start=1):

    print(f"\nSample {i}")


    sample_df = pd.DataFrame([sample])

    predicted_class = best_model.predict(sample_df)[0]
    predicted_probability = best_model.predict_proba(sample_df)[0][predicted_class]

    user_prompt = create_prompt(
        sample,
        predicted_class,
        predicted_probability
    )

    response_temp0 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    response_temp07 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0.7
    )

    print("\nTemperature = 0")
    print(response_temp0)

    print("\nTemperature = 0.7")
    print(response_temp07)

    print("\n")


Sample 1

Temperature = 0
{
  "prediction_label": "Low Risk",
  "confidence_level": "High",
  "top_reason": "Absence of pre-existing conditions and non-smoking status significantly reduce health risk factors",
  "second_reason": "Normal blood pressure (120) and healthy BMI (22.5) indicate good cardiovascular and metabolic health",
  "next_step": "Continue maintaining healthy lifestyle habits including regular exercise and balanced diet"
}

Temperature = 0.7
{
  "prediction_label": "Low Risk",
  "confidence_level": "High",
  "top_reason": "Absence of key risk factors including non-smoker status, no pre-existing conditions, and normal blood pressure (120)",
  "second_reason": "Healthy lifestyle indicators such as normal BMI (22.5), moderate exercise frequency (3), and young age (25)",
  "next_step": "Continue maintaining healthy habits and regular health monitoring"
}



Sample 2

Temperature = 0
{
  "prediction_label": "High Risk",
  "confidence_level": "High",
  "top_reason": "Smoking

In [41]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [42]:
def safe_call_llm(system_prompt, user_prompt, temperature=0):

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt,
        user_prompt,
        temperature=temperature
    )

In [43]:
pii_input = """
Patient email: meena@gmail.com

Age: 30
BMI: 25
"""

response = safe_call_llm(
    system_prompt,
    pii_input,
    temperature=0
)

print(response)

Input blocked: PII detected.
None


In [44]:
clean_input = """
Age: 30
BMI: 25
Non-smoker
Blood Pressure: 120
"""

response = safe_call_llm(
    system_prompt,
    clean_input,
    temperature=0
)

print(response)

{
  "prediction_label": "Low Risk",
  "confidence_level": "High",
  "top_reason": "Normal blood pressure level of 120 mmHg indicates healthy cardiovascular function",
  "second_reason": "Non-smoker status significantly reduces risk of heart disease and other smoking-related conditions",
  "next_step": "Maintain current healthy lifestyle habits and schedule regular health checkups to monitor blood pressure and BMI"
}
